# 🧹 Data Cleaning — Database Q3 2020
> **Mục tiêu:** Clean 3 sheet chính dùng cho dashboard (MKT, Sales, Vận đơn)  
> **Input:** `Database_-_Q3_2020.xlsx`  
> **Output:** `Q3_2020_cleaned.xlsx` gồm 3 sheet đã clean

---
## 📋 Data Issues tìm thấy
| Sheet | Issues chính |
|---|---|
| **MKT** | Inbox dtype=object (mixed), ME1 có 14 nulls, Đơn hàng là float |
| **Sales** | 171 dòng null hoàn toàn · SĐT lưu dạng float · Close Date sai năm · `Type of Lead` viết tắt không chuẩn (Dathang/Tuvan) · `Sales Admin xác nhận` chỉ có 1 giá trị viết hoa DATTHANG · Tên cột tiếng Việt lẫn tiếng Anh không nhất quán |
| **Vận đơn** | 2,124 dòng null (duplicate product rows), nhiều cột không cần thiết |


## 0️⃣ Setup & Import

In [32]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from google.colab import files

uploaded = files.upload()  # Chọn file Excel

# Lấy tên file đầu tiên
file_name = list(uploaded.keys())[0]

print('✅ Libraries loaded')
print('📂 File uploaded:', file_name)

# Hiển thị các sheet trong file
xls = pd.ExcelFile(file_name)
print('📂 Sheets available:', xls.sheet_names)

Saving Database - Q3_2020.xlsx to Database - Q3_2020 (4).xlsx
✅ Libraries loaded
📂 File uploaded: Database - Q3_2020 (4).xlsx
📂 Sheets available: ['MKT', 'Sales', 'Vận đơn', 'Lead-Dathang', 'Mess-CVS', 'Book_CRM_Outsource', 'Book_CRM_LyPhan', 'Mess', 'Mess_Daily_Report', 'Marketer_Paid Revenue 2']


In [40]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from google.colab import files # Ensure files is imported if not already

# Bước 1: Tạo DataFrame mẫu (bạn thay bằng file của bạn)

# Bước 2: Thông tin kết nối BigQuery
project_id = "jda-k1"
dataset_id = "practice_data_pipeline"
table_id = "huuloc_prj2"

# UPLOAD SERVICE ACCOUNT KEY FILE
# files.upload() returns a dictionary of {'filename': b'file_content'}
# We need the filename (the key of the dictionary).
uploaded_key_files = files.upload()
key_path = list(uploaded_key_files.keys())[0] # Extract the filename from the dictionary

# Bước 3: Tạo client BigQuery
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

# Bước 4: Tạo tham chiếu bảng
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# Bước 5: Cấu hình ghi đè bảng nếu đã tồn tại
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    autodetect=True,
)

# Bước 6: Load DataFrame vào BigQuery
# You need to specify which DataFrame you want to upload (e.g., df_mkt, df_sales, df_orders_main)
# The variable 'xls' is a pandas.ExcelFile object, not a DataFrame.
# Replace 'df_mkt' with the DataFrame you intend to upload to this specific table.
# For example, to upload 'df_mkt':
job = client.load_table_from_dataframe(xls, table_ref, job_config=job_config)
job.result()

# Bước 7: Xác nhận
print(f"✅ Đã tải {job.output_rows} dòng vào bảng {table_ref}")

Saving jda-k1-2afad6d8f47e.json to jda-k1-2afad6d8f47e (2).json
✅ Đã tải 1511 dòng vào bảng jda-k1.practice_data_pipeline.huuloc_prj2


---
## 1️⃣ Sheet MKT — Marketing Performance

In [15]:
# --- Load ---
mkt = pd.read_excel(xls, sheet_name='MKT')
print(f'Raw shape: {mkt.shape}')
mkt.head(3)

Raw shape: (1511, 21)


,Date,Channel,MKTer,Chiến dịch,Chi phí Marketing,Impression,Reach,Click,Share,Cmt,...,Lead MKT,Đơn hàng,Doanh thu,Paid Revenue 1,Giá Lead,Đơn/Lead,CPM,CPC,Giá Mess\n(Cmt + Inbox),ME1 (ME đã bao gồm thuế + phí)
0,2020-07-01,FB,DucVu,CVS,0,0,0.0,0,0,0,...,1,1.0,499000.0,499000,0.0,1.000000,0.000000,0.00000,0.0,0.00
1,2020-07-01,FB,DucVu,Mess,0,0,0.0,0,0,0,...,3,1.0,499000.0,499000,0.0,0.333333,0.000000,0.00000,0.0,0.00
2,2020-07-01,FB,DucVu,Brand,259352,61673,59968.0,271,0,0,...,0,0.0,0.0,0,0.0,0.000000,4205.276215,957.01845,0.0,280100.16


In [16]:
# --- Audit ---
print('=== DTYPES ===')
print(mkt.dtypes)
print('\n=== NULL COUNT ===')
print(mkt.isnull().sum())
print('\n=== INBOX sample (mixed type) ===')
print(mkt['Inbox'].value_counts().head(10))

=== DTYPES ===
Date                              datetime64[ns]
Channel                                   object
MKTer                                     object
Chiến dịch                                object
Chi phí Marketing                          int64
Impression                                 int64
Reach                                    float64
Click                                      int64
Share                                      int64
Cmt                                        int64
Inbox                                     object
Lead MKT                                   int64
Đơn hàng                                 float64
Doanh thu                                float64
Paid Revenue 1                             int64
Giá Lead                                 float64
Đơn/Lead                                 float64
CPM                                      float64
CPC                                      float64
Giá Mess\n(Cmt + Inbox)                  float64
ME1 (

In [17]:
# ============================================================
# CLEAN MKT
# ============================================================
df_mkt = mkt.copy()

# 1. Rename columns — bỏ ký tự đặc biệt và xuống dòng
df_mkt.columns = [
    'Date', 'Channel', 'MKTer', 'Campaign',
    'Spend', 'Impression', 'Reach', 'Click',
    'Share', 'Comment', 'Inbox', 'Lead',
    'Order', 'Revenue', 'Paid_Revenue_1',
    'CPL', 'Order_per_Lead', 'CPM', 'CPC',
    'Cost_per_Mess', 'ME1'
]
print('✅ Columns renamed')

# 2. Date — đảm bảo datetime
df_mkt['Date'] = pd.to_datetime(df_mkt['Date'], errors='coerce')
print(f'✅ Date range: {df_mkt["Date"].min().date()} → {df_mkt["Date"].max().date()}')

# 3. Inbox — có giá trị 'Cộng tác viên', số, NaN → coerce về numeric
df_mkt['Inbox'] = pd.to_numeric(df_mkt['Inbox'], errors='coerce').fillna(0).astype(int)
print(f'✅ Inbox cleaned — unique non-zero: {df_mkt["Inbox"].nunique()}')

# 4. Order — float → int (không có lý do để là float)
df_mkt['Order'] = df_mkt['Order'].fillna(0).astype(int)
print('✅ Order → int')

# 5. ME1 — 14 nulls → fill bằng 0 (ngày không có data)
null_me1 = df_mkt['ME1'].isnull().sum()
df_mkt['ME1'] = df_mkt['ME1'].fillna(0)
print(f'✅ ME1: filled {null_me1} nulls → 0')

# 6. Thêm cột Month, Week để dùng trong dashboard
df_mkt['Month'] = df_mkt['Date'].dt.to_period('M').astype(str)
df_mkt['Week'] = df_mkt['Date'].dt.to_period('W').astype(str)
print('✅ Month & Week columns added')

# 7. Tính lại ROAS (Revenue / Spend) — tránh division by zero
df_mkt['ROAS'] = np.where(
    df_mkt['Spend'] > 0,
    (df_mkt['Revenue'] / df_mkt['Spend']).round(2),
    0
)
print('✅ ROAS column recalculated')

# 8. CTR
df_mkt['CTR'] = np.where(
    df_mkt['Impression'] > 0,
    (df_mkt['Click'] / df_mkt['Impression'] * 100).round(4),
    0
)
print('✅ CTR column added')

print(f'\n📊 MKT clean shape: {df_mkt.shape}')
df_mkt.head(3)

✅ Columns renamed
✅ Date range: 2020-07-01 → 2020-09-30
✅ Inbox cleaned — unique non-zero: 3
✅ Order → int
✅ ME1: filled 14 nulls → 0
✅ Month & Week columns added
✅ ROAS column recalculated
✅ CTR column added

📊 MKT clean shape: (1511, 25)


,Date,Channel,MKTer,Campaign,Spend,Impression,Reach,Click,Share,Comment,...,CPL,Order_per_Lead,CPM,CPC,Cost_per_Mess,ME1,Month,Week,ROAS,CTR
0,2020-07-01,FB,DucVu,CVS,0,0,0.0,0,0,0,...,0.0,1.000000,0.000000,0.00000,0.0,0.00,2020-07,2020-06-29/2020-07-05,0.0,0.0000
1,2020-07-01,FB,DucVu,Mess,0,0,0.0,0,0,0,...,0.0,0.333333,0.000000,0.00000,0.0,0.00,2020-07,2020-06-29/2020-07-05,0.0,0.0000
2,2020-07-01,FB,DucVu,Brand,259352,61673,59968.0,271,0,0,...,0.0,0.000000,4205.276215,957.01845,0.0,280100.16,2020-07,2020-06-29/2020-07-05,0.0,0.4394


In [18]:
# --- Verify MKT ---
print('=== MKT SUMMARY ===')
print(f'Total spend:   {df_mkt["Spend"].sum():>15,.0f} VND')
print(f'Total revenue: {df_mkt["Revenue"].sum():>15,.0f} VND')
print(f'Overall ROAS:  {df_mkt["Revenue"].sum() / df_mkt["Spend"].sum():>14.2f}x')
print(f'Total leads:   {df_mkt["Lead"].sum():>15,.0f}')
print(f'Total orders:  {df_mkt["Order"].sum():>15,.0f}')
print(f'Nulls left:    {df_mkt.isnull().sum().sum()}')

=== MKT SUMMARY ===
Total spend:     1,447,025,015 VND
Total revenue:   3,440,747,612 VND
Overall ROAS:            2.38x
Total leads:             9,261
Total orders:            6,490
Nulls left:    0


---
## 2️⃣ Sheet Sales — Lead & Conversion Data

In [20]:
# --- Load ---
sales = pd.read_excel(xls, sheet_name='Sales')
print(f'Raw shape: {sales.shape}')
sales.head(3)

Raw shape: (9675, 21)


,Unnamed: 0,Giờ,Khách hàng,SĐT,Channel,Chiến dịch,Content,Marketer 2,Type of Lead,Sales Admin xác nhận Type of Lead,...,Số lần tương tác,Ngày gọi,Trạng thái,Level,Ngày hẹn gọi lại,Close Date,Tỉnh/TP,Số lượng bộ sách,Số tiền giảm giá,Tổng tiền
0,1-7-2020,2023-07-01 00:13:42,Pham thu thuy,123400000.0,FB,CVS,B1-1812-P1,AnhNguyen08,Dathang,DATTHANG,...,1.0,2020-07-01,Hủy đơn sau khi chốt,L7.2 - Hủy đơn sau khi chốt,NaT,NaT,NaN,1.0,140000.0,499000
1,1-7-2020,2023-07-01 00:17:53,Hoàng Thị Lương,123400001.0,FB,CVS,cay,NgocNgo,Dathang,DATTHANG,...,1.0,2020-07-01,Đặt hàng,L8.2 - Khách đã thanh toán toàn bộ,NaT,2023-07-04,Quảng Ninh,1.0,140000.0,499000
2,1-7-2020,2023-07-01 02:16:56,Phạm Thị huong,123400002.0,FB,CVS,matcon,NgocNgo,Dathang,DATTHANG,...,1.0,2020-07-03,Hủy đơn sau khi chốt,L7.2 - Hủy đơn sau khi chốt,NaT,NaT,NaN,1.0,140000.0,499000


In [21]:
# --- Audit Sales ---
print('=== NULL COUNTS ===')
null_pct = (sales.isnull().sum() / len(sales) * 100).round(1)
for col, pct in null_pct[null_pct > 0].items():
    print(f'  {col:<45} {pct:>5}% null')

print('\n=== [1] TYPE OF LEAD — viết tắt không chuẩn ===')
print(sales['Type of Lead'].value_counts(dropna=False))
# Dathang  → Đặt hàng
# Tuvan    → Tư vấn
# (1 dòng) 'Con đang học lớp 7' → sai, cần flag

print('\n=== [2] SALES ADMIN XÁC NHẬN — viết hoa không chuẩn ===')
print(sales['Sales Admin xác nhận Type of Lead'].value_counts(dropna=False))
# DATTHANG → Đặt hàng (đồng nhất với Type of Lead)

print('\n=== [3] TÊN CỘT chưa chuẩn ===')
for i, col in enumerate(sales.columns):
    print(f'  [{i:>2}] {repr(col)}')

print('\n=== [4] CLOSE DATE year distribution ===')
print(pd.to_datetime(sales['Close Date'], errors='coerce').dt.year.value_counts())

print('\n=== [5] SĐT sample (stored as float) ===')
print(sales['SĐT'].head(5).tolist())

print('\n=== [6] CHANNEL — nguồn nào xuất hiện ===')
print(sales['Channel'].value_counts(dropna=False))

=== NULL COUNTS ===
  Unnamed: 0                                      1.8% null
  Giờ                                             1.8% null
  Khách hàng                                      1.8% null
  SĐT                                             1.8% null
  Channel                                         1.8% null
  Chiến dịch                                      1.8% null
  Content                                         1.8% null
  Marketer 2                                     29.9% null
  Type of Lead                                    1.8% null
  Sales Admin xác nhận Type of Lead              43.7% null
  Sales                                           1.8% null
  Số lần tương tác                                1.9% null
  Ngày gọi                                       97.9% null
  Trạng thái                                      3.7% null
  Level                                          17.5% null
  Ngày hẹn gọi lại                               99.5% null
  Close Date        

In [22]:
# ============================================================
# CLEAN SALES — bao gồm chuẩn hóa giá trị viết tắt & tên cột
# ============================================================
df_sales = sales.copy()

# ── BƯỚC 1: Drop 171 dòng NULL hoàn toàn ────────────────────
before = len(df_sales)
df_sales = df_sales.dropna(subset=['Unnamed: 0', 'Khách hàng', 'SĐT', 'Channel', 'Sales'])
print(f'✅ [1] Dropped {before - len(df_sales)} fully-null rows | Remaining: {len(df_sales)}')

# ── BƯỚC 2: Rename cột — chuẩn hóa toàn bộ sang snake_case nhất quán ──
# Nguyên tắc: tiếng Anh, snake_case, rõ nghĩa, không dấu
df_sales.columns = [
    'lead_id',            # Unnamed: 0  → mã lead theo ngày
    'lead_time',          # Giờ         → giờ nhận lead
    'customer_name',      # Khách hàng
    'phone',              # SĐT
    'channel',            # Channel
    'campaign',           # Chiến dịch
    'ad_content',         # Content     → nội dung quảng cáo
    'marketer',           # Marketer 2
    'lead_type',          # Type of Lead
    'lead_type_confirmed',# Sales Admin xác nhận Type of Lead
    'salesperson',        # Sales
    'interaction_count',  # Số lần tương tác
    'call_date',          # Ngày gọi
    'status',             # Trạng thái
    'level',              # Level
    'followup_date',      # Ngày hẹn gọi lại
    'close_date',         # Close Date
    'province',           # Tỉnh/TP
    'book_qty',           # Số lượng bộ sách
    'discount',           # Số tiền giảm giá
    'total_amount'        # Tổng tiền
]
print('✅ [2] Columns renamed → snake_case nhất quán')
print('   Mapping nổi bật:')
print('     Unnamed: 0                          → lead_id')
print('     Type of Lead                        → lead_type')
print('     Sales Admin xác nhận Type of Lead   → lead_type_confirmed')
print('     Marketer 2                          → marketer')
print('     Tỉnh/TP                             → province')
print('     Content                             → ad_content')

# ── BƯỚC 3: Chuẩn hóa lead_type — viết tắt → full text ─────
# Giá trị gốc: 'Dathang', 'Tuvan', 'Con đang học lớp 7' (data lỗi)
print(f'\n📋 [3] lead_type trước khi clean:')
print(df_sales['lead_type'].value_counts(dropna=False))

lead_type_map = {
    'Dathang'              : 'Đặt hàng',   # viết tắt không dấu → full
    'Tuvan'                : 'Tư vấn',     # viết tắt không dấu → full
    'Con đang học lớp 7'   : None,         # data lỗi (câu trả lời khách), gán null
}
df_sales['lead_type'] = df_sales['lead_type'].map(
    lambda x: lead_type_map.get(x, x) if pd.notna(x) else x
)
print(f'\n✅ lead_type sau khi clean:')
print(df_sales['lead_type'].value_counts(dropna=False))

# ── BƯỚC 4: Chuẩn hóa lead_type_confirmed ───────────────────
# Giá trị gốc: 'DATTHANG' (viết hoa toàn bộ, viết tắt)
# → đồng nhất với lead_type: 'Đặt hàng'
print(f'\n📋 [4] lead_type_confirmed trước khi clean:')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

confirmed_map = {
    'DATTHANG': 'Đặt hàng',   # ALLCAPS viết tắt → full text, đồng nhất với lead_type
}
df_sales['lead_type_confirmed'] = df_sales['lead_type_confirmed'].map(
    lambda x: confirmed_map.get(str(x).strip(), x) if pd.notna(x) else x
)
print(f'\n✅ lead_type_confirmed sau khi clean:')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

# ── BƯỚC 5: Phone — float → string ──────────────────────────
df_sales['phone'] = df_sales['phone'].apply(
    lambda x: str(int(x)) if pd.notna(x) and x != 0 else None
)
print('\n✅ [5] phone cleaned: float → string')

# ── BƯỚC 6: Dates ────────────────────────────────────────────
# close_date — sửa năm sai (2023 → 2020, do Excel date serialization)
df_sales['close_date'] = pd.to_datetime(df_sales['close_date'], errors='coerce')
mask_wrong_year = df_sales['close_date'].dt.year.isin([2023, 2024, 2025])
n_wrong = mask_wrong_year.sum()
df_sales.loc[mask_wrong_year, 'close_date'] = (
    df_sales.loc[mask_wrong_year, 'close_date']
    .apply(lambda x: x.replace(year=2020) if pd.notna(x) else x)
)
print(f'✅ [6] close_date: fixed {n_wrong} wrong-year entries → 2020')
df_sales['call_date'] = pd.to_datetime(df_sales['call_date'], errors='coerce')
df_sales['followup_date'] = pd.to_datetime(df_sales['followup_date'], errors='coerce')
print('✅ [6] call_date, followup_date → datetime')

# ── BƯỚC 7: Thêm status_group (gom nhóm funnel) ─────────────
status_map = {
    'Đặt hàng'                 : 'Chuyển đổi',
    'Hủy đơn sau khi chốt'     : 'Hủy',
    'Từ chối'                  : 'Từ chối',
    'Suy nghĩ thêm/Trả lời sau': 'Đang xử lý',
    'Hẹn lúc khác gọi lại'     : 'Đang xử lý',
    'Không nghe máy'           : 'Không liên lạc được',
    'Thuê bao'                 : 'Không liên lạc được',
    'Sai số'                   : 'Không hợp lệ',
    'Trùng'                    : 'Không hợp lệ',
    'Trêu đùa'                 : 'Không hợp lệ',
}
df_sales['status'] = df_sales['status'].str.strip()
df_sales['status_group'] = df_sales['status'].map(status_map).fillna('Chưa phân loại')
print('✅ [7] status_group column added (tiếng Việt, 6 nhóm)')

# ── BƯỚC 8: Numeric cleanup ──────────────────────────────────
df_sales['book_qty']  = df_sales['book_qty'].fillna(0).astype(int)
df_sales['discount']  = df_sales['discount'].fillna(0)
df_sales['province']  = df_sales['province'].str.strip()
df_sales['level']     = df_sales['level'].str.strip()
print('✅ [8] book_qty, discount nulls → 0 | province/level stripped')

print(f'\n📊 Sales clean shape: {df_sales.shape}')
df_sales[['lead_type','lead_type_confirmed','status','status_group','channel']].head(10)

✅ [1] Dropped 171 fully-null rows | Remaining: 9504
✅ [2] Columns renamed → snake_case nhất quán
   Mapping nổi bật:
     Unnamed: 0                          → lead_id
     Type of Lead                        → lead_type
     Sales Admin xác nhận Type of Lead   → lead_type_confirmed
     Marketer 2                          → marketer
     Tỉnh/TP                             → province
     Content                             → ad_content

📋 [3] lead_type trước khi clean:
lead_type
Dathang               6233
Tuvan                 3264
NaN                      6
Con đang học lớp 7       1
Name: count, dtype: int64

✅ lead_type sau khi clean:
lead_type
Đặt hàng    6233
Tư vấn      3264
NaN            6
None           1
Name: count, dtype: int64

📋 [4] lead_type_confirmed trước khi clean:
lead_type_confirmed
DATTHANG    5448
NaN         4056
Name: count, dtype: int64

✅ lead_type_confirmed sau khi clean:
lead_type_confirmed
Đặt hàng    5448
NaN         4056
Name: count, dtype: int64

✅ [5]

,lead_type,lead_type_confirmed,status,status_group,channel
0,Đặt hàng,Đặt hàng,Hủy đơn sau khi chốt,Hủy,FB
1,Đặt hàng,Đặt hàng,Đặt hàng,Chuyển đổi,FB
2,Đặt hàng,Đặt hàng,Hủy đơn sau khi chốt,Hủy,FB
3,Đặt hàng,Đặt hàng,Hủy đơn sau khi chốt,Hủy,FB
4,Đặt hàng,Đặt hàng,Đặt hàng,Chuyển đổi,FB
5,Đặt hàng,Đặt hàng,Đặt hàng,Chuyển đổi,FB
6,Đặt hàng,Đặt hàng,Hủy đơn sau khi chốt,Hủy,FB
7,Đặt hàng,Đặt hàng,Đặt hàng,Chuyển đổi,FB
8,Đặt hàng,Đặt hàng,Đặt hàng,Chuyển đổi,FB
9,Đặt hàng,Đặt hàng,Hủy đơn sau khi chốt,Hủy,FB


In [23]:
# --- Verify Sales ---
print('=== SALES SUMMARY ===')
print(f'Total leads:  {len(df_sales):>10,}')

print('\n── lead_type (sau chuẩn hóa) ──')
print(df_sales['lead_type'].value_counts(dropna=False))

print('\n── lead_type_confirmed (sau chuẩn hóa) ──')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

print('\n── status_group breakdown ──')
sg = df_sales['status_group'].value_counts()
for k, v in sg.items():
    print(f'  {k:<25} {v:>6,}  ({v/len(df_sales)*100:.1f}%)')

print(f'\nConversion rate: {(df_sales["status_group"]=="Chuyển đổi").mean()*100:.1f}%')

print('\n── close_date year after fix ──')
print(df_sales['close_date'].dt.year.value_counts())

print('\n── Nulls in key columns ──')
key_cols = ['lead_id','customer_name','channel','campaign','lead_type','lead_type_confirmed','status','status_group']
print(df_sales[key_cols].isnull().sum())

=== SALES SUMMARY ===
Total leads:       9,504

── lead_type (sau chuẩn hóa) ──
lead_type
Đặt hàng    6233
Tư vấn      3264
NaN            6
None           1
Name: count, dtype: int64

── lead_type_confirmed (sau chuẩn hóa) ──
lead_type_confirmed
Đặt hàng    5448
NaN         4056
Name: count, dtype: int64

── status_group breakdown ──
  Chuyển đổi                 6,343  (66.7%)
  Đang xử lý                 1,409  (14.8%)
  Hủy                          533  (5.6%)
  Không liên lạc được          498  (5.2%)
  Từ chối                      430  (4.5%)
  Chưa phân loại               186  (2.0%)
  Không hợp lệ                 105  (1.1%)

Conversion rate: 66.7%

── close_date year after fix ──
close_date
2020.0    6161
Name: count, dtype: int64

── Nulls in key columns ──
lead_id                   0
customer_name             0
channel                   0
campaign                  0
lead_type                 7
lead_type_confirmed    4056
status                  186
status_group              0

---
## 3️⃣ Sheet Vận đơn — Orders & Delivery

In [25]:
# --- Load ---
orders = pd.read_excel(xls, sheet_name='Vận đơn')
print(f'Raw shape: {orders.shape}')
orders.head(4)

Raw shape: (7944, 45)


,STT,Mã đơn hàng,Ghi chú đơn hàng,Tags đơn hàng,Nhân viên tạo đơn,Chi nhánh,Nguồn,Mã vận đơn,Tình trạng gói hàng,Trạng thái đối tác,...,Tên sản phẩm,Ghi chú sản phẩm,Số lượng,Serial,Đơn vị tính,Đơn giá,CK sản phẩm,CK tổng đơn hàng,Thuế cho từng sản phẩm,Tổng tiền hàng
0,1.0,SON25144,NaN,NaN,Nguyễn Thị Huyền,Chi nhánh mặc định,Web,G87L96F3,Đã giao hàng,Giao thành công,...,Sách TeenUp - PH,NaN,1,NaN,NaN,0,0,0,0,0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BỘ SÁCH TEENUP - KID,NaN,1,NaN,NaN,499000,0,0,0,499000
2,2.0,SON24883,TẶNG SÁCH LẺ PHẦN 2 CHO PHỤ HUỲNH\ngiao trong ...,NaN,Đỗ Thị Hồng Ngọc,Chi nhánh mặc định,Web,G87L9VTM,Đã giao hàng,Giao thành công,...,Sách TeenUp - Phần 2,NaN,1,NaN,NaN,0,0,0,0,0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BỘ SÁCH TEENUP - TEEN,NaN,1,NaN,Bộ,639000,140000,0,0,499000


In [26]:
# --- Audit Vận đơn ---
print('=== NULL % per column ===')
null_pct = (orders.isnull().sum() / len(orders) * 100).round(1)
for col, pct in null_pct[null_pct > 0].items():
    print(f'  {col:<35} {pct:>6}%')

print('\n=== Delivery status ===')
print(orders['Tình trạng gói hàng'].value_counts())

print('\n=== Delivery partner ===')
print(orders['Đối tác giao hàng'].value_counts())

=== NULL % per column ===
  STT                                   26.7%
  Mã đơn hàng                           26.7%
  Ghi chú đơn hàng                      92.9%
  Tags đơn hàng                         99.7%
  Nhân viên tạo đơn                     26.7%
  Chi nhánh                             26.7%
  Nguồn                                 26.7%
  Mã vận đơn                            26.7%
  Tình trạng gói hàng                   26.7%
  Trạng thái đối tác                    26.7%
  Lý do hủy đơn                        100.0%
  Ngày đóng gói                         26.7%
  Ngày hẹn giao                        100.0%
  Ngày xuất kho                         26.7%
  Ngày giao hàng                        35.6%
  Đối tác giao hàng                     26.7%
  Dịch vụ giao hàng                     26.7%
  Khối lượng(g)                         26.7%
  Kích thước(DxRxC)                     26.7%
  Tên người nhận                        26.7%
  SĐT người nhận                        26.7%
  Địa ch

In [27]:
# ============================================================
# CLEAN VẬN ĐƠN
# ============================================================
df_orders = orders.copy()

# 1. Xác định dòng "header" của order (có STT) vs dòng product detail
# Mỗi đơn hàng có thể có nhiều dòng sản phẩm. Dòng đầu tiên có STT là header.
# Giữ cả hai loại nhưng đánh dấu rõ
df_orders['Is_Order_Header'] = df_orders['STT'].notna()
print(f'Order headers (unique orders): {df_orders["Is_Order_Header"].sum():,}')
print(f'Product detail rows:           {(~df_orders["Is_Order_Header"]).sum():,}')

# 2. Tạo df_orders_main — chỉ lấy dòng order header (dùng cho dashboard level)
df_orders_main = df_orders[df_orders['Is_Order_Header']].copy()
print(f'\n✅ Main orders extracted: {len(df_orders_main)} rows')

# 3. Rename các cột quan trọng
rename_map = {
    'STT': 'Order_No',
    'Mã đơn hàng': 'Order_ID',
    'Tình trạng gói hàng': 'Delivery_Status',
    'Trạng thái đối tác': 'Partner_Status',
    'Ngày đóng gói': 'Pack_Date',
    'Ngày xuất kho': 'Dispatch_Date',
    'Ngày giao hàng': 'Delivery_Date',
    'Đối tác giao hàng': 'Courier',
    'Dịch vụ giao hàng': 'Courier_Service',
    'Tên người nhận': 'Recipient_Name',
    'SĐT người nhận': 'Recipient_Phone',
    'Tỉnh/Thành': 'Province',
    'Quận/Huyện': 'District',
    'Tiền khách phải trả cho đơn': 'Amount_Due',
    'Khách hàng đã trả': 'Amount_Paid',
    'Hình thức thanh toán': 'Payment_Method',
    'Phí vận chuyển': 'Shipping_Fee',
    'Tổng tiền hàng': 'Total_Amount',
    'Số lượng': 'Qty',
    'Đơn giá': 'Unit_Price'
}
df_orders_main = df_orders_main.rename(columns=rename_map)
print('✅ Columns renamed')

# 4. Dates → datetime
for col in ['Pack_Date', 'Dispatch_Date', 'Delivery_Date']:
    if col in df_orders_main.columns:
        df_orders_main[col] = pd.to_datetime(df_orders_main[col], errors='coerce')
print('✅ Date columns → datetime')

# 5. Tính Fulfillment_Days (xuất kho → giao hàng)
df_orders_main['Fulfillment_Days'] = (
    df_orders_main['Delivery_Date'] - df_orders_main['Dispatch_Date']
).dt.days
print(f'✅ Fulfillment_Days added | avg: {df_orders_main["Fulfillment_Days"].mean():.1f} days')

# 6. Delivery_Status — chuẩn hóa
df_orders_main['Delivery_Status'] = df_orders_main['Delivery_Status'].str.strip()

# 7. Thêm Delivery_Success flag
df_orders_main['Delivery_Success'] = df_orders_main['Delivery_Status'] == 'Đã giao hàng'

# 8. Province — strip
df_orders_main['Province'] = df_orders_main['Province'].str.strip()

# 9. Chỉ giữ cột cần thiết cho dashboard
keep_cols = [
    'Order_No', 'Order_ID', 'Delivery_Status', 'Partner_Status',
    'Pack_Date', 'Dispatch_Date', 'Delivery_Date', 'Fulfillment_Days',
    'Courier', 'Courier_Service', 'Province', 'District',
    'Amount_Due', 'Amount_Paid', 'Payment_Method', 'Shipping_Fee',
    'Total_Amount', 'Qty', 'Unit_Price', 'Delivery_Success'
]
# Chỉ giữ cột tồn tại
keep_cols = [c for c in keep_cols if c in df_orders_main.columns]
df_orders_main = df_orders_main[keep_cols]

print(f'\n📊 Orders clean shape: {df_orders_main.shape}')
df_orders_main.head(3)

Order headers (unique orders): 5,820
Product detail rows:           2,124

✅ Main orders extracted: 5820 rows
✅ Columns renamed
✅ Date columns → datetime
✅ Fulfillment_Days added | avg: 1.6 days

📊 Orders clean shape: (5820, 20)


,Order_No,Order_ID,Delivery_Status,Partner_Status,Pack_Date,Dispatch_Date,Delivery_Date,Fulfillment_Days,Courier,Courier_Service,Province,District,Amount_Due,Amount_Paid,Payment_Method,Shipping_Fee,Total_Amount,Qty,Unit_Price,Delivery_Success
0,1.0,SON25144,Đã giao hàng,Giao thành công,2020-09-16 12:49:54,2020-09-17 11:05:38,2020-09-19 10:20:38,1.0,Sapo Express,Chuẩn,Hưng Yên,Huyện Văn Lâm,499000.0,499000.0,COD,0.0,0,1,0,True
2,2.0,SON24883,Đã giao hàng,Giao thành công,2020-09-16 12:49:13,2020-09-17 11:05:48,2020-09-21 12:18:59,4.0,Sapo Express,Chuẩn,TP Hồ Chí Minh,Quận 7,499000.0,499000.0,COD,0.0,0,1,0,True
4,3.0,SON24943,Đã giao hàng,Giao thành công,2020-09-15 19:56:20,2020-09-15 19:58:37,2020-09-19 12:54:17,3.0,Sapo Express,(Ninja Van - Chuẩn),Hà Nội,Quận Hoàn Kiếm,780000.0,780000.0,COD,0.0,780000,1,920000,True


In [28]:
# --- Verify Orders ---
print('=== ORDERS SUMMARY ===')
print(f'Total orders:         {len(df_orders_main):>8,}')
print(f'Delivered:            {df_orders_main["Delivery_Success"].sum():>8,} ({df_orders_main["Delivery_Success"].mean()*100:.1f}%)')
print(f'\nDelivery status breakdown:')
print(df_orders_main['Delivery_Status'].value_counts())
print(f'\nAvg fulfillment days: {df_orders_main["Fulfillment_Days"].mean():.1f}')
print(f'Top 3 provinces:')
print(df_orders_main['Province'].value_counts().head(3))

=== ORDERS SUMMARY ===
Total orders:            5,820
Delivered:               5,115 (87.9%)

Delivery status breakdown:
Delivery_Status
Đã giao hàng        5115
Hủy giao-đã nhận     701
Đang giao hàng         2
Chờ giao lại           2
Name: count, dtype: int64

Avg fulfillment days: 1.6
Top 3 provinces:
Province
Hà Nội            1642
TP Hồ Chí Minh    1075
Hải Phòng          191
Name: count, dtype: int64


---
## 4️⃣ Export — Lưu file đã clean

In [29]:
OUTPUT = 'Q3_2020_cleaned.xlsx'

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    df_mkt.to_excel(writer, sheet_name='MKT_clean', index=False)
    df_sales.to_excel(writer, sheet_name='Sales_clean', index=False)
    df_orders_main.to_excel(writer, sheet_name='Orders_clean', index=False)

print(f'✅ Exported: {OUTPUT}')
print(f'   MKT_clean:    {df_mkt.shape}')
print(f'   Sales_clean:  {df_sales.shape}')
print(f'   Orders_clean: {df_orders_main.shape}')

✅ Exported: Q3_2020_cleaned.xlsx
   MKT_clean:    (1511, 25)
   Sales_clean:  (9504, 22)
   Orders_clean: (5820, 20)


In [30]:
# Download file (chạy trên Google Colab)
from google.colab import files
files.download(OUTPUT)
print('📥 Uncomment 2 dòng trên nếu đang dùng Google Colab để download file')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Uncomment 2 dòng trên nếu đang dùng Google Colab để download file


---
## 5️⃣ Quick Sanity Check — Cross-sheet Validation

In [31]:
print('='*55)
print('CROSS-SHEET SANITY CHECK')
print('='*55)

# Leads so sánh
leads_mkt = df_mkt['Lead'].sum()
leads_sales = len(df_sales)
print(f'\n📌 Leads:')
print(f'  MKT sheet reported:  {leads_mkt:>8,.0f}')
print(f'  Sales sheet rows:    {leads_sales:>8,}')
print(f'  Gap:                 {leads_sales - leads_mkt:>+8,.0f} ({(leads_sales-leads_mkt)/leads_mkt*100:+.1f}%)')
print('  → Bình thường (Sales ghi thêm leads organic / referral không qua MKT)')

# Orders so sánh
orders_mkt = df_mkt['Order'].sum()
orders_sales = (df_sales['status_group'] == 'Chuyển đổi').sum()
orders_system = len(df_orders_main)
print(f'\n📌 Orders:')
print(f'  MKT sheet reported:  {orders_mkt:>8,.0f}')
print(f'  Sales converted:     {orders_sales:>8,}')
print(f'  Order system (Vận đơn): {orders_system:>5,}')
print('  → Vận đơn lớn hơn vì có đơn từ nguồn khác ngoài MKT tracked')

# Revenue so sánh
rev_mkt = df_mkt['Revenue'].sum()
rev_orders = df_orders_main['Amount_Due'].sum()
print(f'\n📌 Revenue:')
print(f'  MKT attributed:  {rev_mkt:>15,.0f} VND')
print(f'  Orders amount:   {rev_orders:>15,.0f} VND')

print('\n✅ Sanity check complete — data sẵn sàng cho dashboard!')

CROSS-SHEET SANITY CHECK

📌 Leads:
  MKT sheet reported:     9,261
  Sales sheet rows:       9,504
  Gap:                     +243 (+2.6%)
  → Bình thường (Sales ghi thêm leads organic / referral không qua MKT)

📌 Orders:
  MKT sheet reported:     6,490
  Sales converted:        6,343
  Order system (Vận đơn): 5,820
  → Vận đơn lớn hơn vì có đơn từ nguồn khác ngoài MKT tracked

📌 Revenue:
  MKT attributed:    3,440,747,612 VND
  Orders amount:     2,918,878,600 VND

✅ Sanity check complete — data sẵn sàng cho dashboard!


---
## 📝 Tóm tắt thao tác đã làm

| Sheet | Thao tác |
|---|---|
| **MKT** | Rename 21 cột · Fix Inbox dtype (object→int) · Fill ME1 nulls · Fix Order (float→int) · Thêm Month/Week/ROAS/CTR |
| **Sales** | Drop 171 null rows · Rename 21 cột → snake_case · `lead_type`: Dathang→Đặt hàng, Tuvan→Tư vấn, data lỗi→null · `lead_type_confirmed`: DATTHANG→Đặt hàng · Phone float→string · Fix close_date year 2023→2020 · Thêm status_group (6 nhóm tiếng Việt) · Fill book_qty/discount |
| **Vận đơn** | Tách header rows vs product rows · Rename cột · Parse dates · Tính Fulfillment_Days · Thêm Delivery_Success flag · Giữ 20 cột cần thiết |

> **File output:** `Q3_2020_cleaned.xlsx` — 3 sheets sạch, sẵn sàng kéo vào Power BI / Looker Studio / Tableau.
